In [ ]:
!pip install -q "transformers==4.40.0" "peft==0.10.0" accelerate datasets evaluate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 78.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.6 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [ ]:
import os, random
import pandas as pd
import numpy as np
import torch
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
)
from torch.utils.data import Dataset
from torch.nn import CrossEntropyLoss

# Fix all seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("transformers:", __import__("transformers").__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


2026-03-14 18:45:27.144874: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773513927.387089      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773513927.451536      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773513927.982584      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773513927.982629      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773513927.982632      55 computation_placer.cc:177] computation placer alr

transformers: 4.40.0
CUDA: True
GPU: Tesla T4


In [ ]:
import os
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/daieeane/train6/sample_submission.csv
/kaggle/input/datasets/daieeane/train6/train_example.csv
/kaggle/input/datasets/daieeane/train6/test.csv
/kaggle/input/datasets/daieeane/train6/train_final_v6.csv
/kaggle/input/competitions/kaz-punct-hackathon/sample_submission.csv
/kaggle/input/competitions/kaz-punct-hackathon/train_example.csv
/kaggle/input/competitions/kaz-punct-hackathon/test.csv


## Data
Two CSV files merged into ~100k rows. Each row: a sentence of whitespace-split words + corresponding labels.  
Class distribution is heavily imbalanced — QUESTION is only 0.3% of tokens.

In [ ]:
df_train  = pd.read_csv("/kaggle/input/datasets/daieeane/train6/train_final_v6.csv")
df_example = pd.read_csv("/kaggle/input/datasets/daieeane/train6/train_example.csv")

df_all = pd.concat([df_train, df_example], ignore_index=True)
df_all = df_all.dropna(subset=["input_text", "labels"])
df_all["input_text"] = df_all["input_text"].astype(str)
df_all["labels"]     = df_all["labels"].astype(str)
print(f"Total rows: {len(df_all)}")

# Label distribution
all_labels = []
for row in df_all["labels"]:
    all_labels.extend(row.split())
c = Counter(all_labels)
total = sum(c.values())
for k, v in c.most_common():
    print(f"  {k}: {v} ({v/total*100:.1f}%)")


Total rows: 100500
  O: 1635648 (85.6%)
  PERIOD: 138324 (7.2%)
  COMMA: 130271 (6.8%)
  QUESTION: 6434 (0.3%)


## Model & Tokenization
Stratify by first label of each row to keep class distribution in val

In [ ]:
MODEL_NAME = "xlm-roberta-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

LABEL2ID = {"O": 0, "COMMA": 1, "PERIOD": 2, "QUESTION": 3}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)  # 4 — avoids DataParallel .config bug

class PunctuationDataset(Dataset):
    def __init__(self, df, max_length=192):
        self.samples = []
        skipped = 0

        for _, row in df.iterrows():
            words  = row["input_text"].split()
            labels = row["labels"].split()

            # Skip malformed rows
            if len(words) != len(labels):
                skipped += 1
                continue
            if not all(l in LABEL2ID for l in labels):
                skipped += 1
                continue

            encoding = tokenizer(
                words,
                is_split_into_words=True,
                truncation=True,
                max_length=max_length,
                padding="max_length",
            )

            # Align labels to subword tokens; mask non-first subwords
            word_ids       = encoding.word_ids()
            aligned_labels = []
            prev_word_id   = None
            for word_id in word_ids:
                if word_id is None:
                    aligned_labels.append(-100)
                elif word_id != prev_word_id:
                    aligned_labels.append(LABEL2ID[labels[word_id]])
                else:
                    aligned_labels.append(-100)
                prev_word_id = word_id

            self.samples.append({
                "input_ids":      torch.tensor(encoding["input_ids"]),
                "attention_mask": torch.tensor(encoding["attention_mask"]),
                "labels":         torch.tensor(aligned_labels),
            })

        print(f"  Loaded: {len(self.samples)}, skipped: {skipped}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
df_all["_strat"] = df_all["labels"].apply(lambda x: x.split()[0])
train_df, val_df = train_test_split(df_all, test_size=0.05, random_state=SEED, stratify=df_all["_strat"])
df_all.drop(columns=["_strat"], inplace=True)
train_df = train_df.drop(columns=["_strat"])
val_df   = val_df.drop(columns=["_strat"])

print("Preparing train...")
train_dataset = PunctuationDataset(train_df)
print("Preparing val...")
val_dataset   = PunctuationDataset(val_df)


Preparing train...
  Loaded: 95475, skipped: 0
Preparing val...
  Loaded: 5025, skipped: 0


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Training
**Key design decisions:**
- Weighted CrossEntropyLoss: `[O=1.0, COMMA=4.0, PERIOD=2.5, QUESTION=40.0]` to handle class imbalance
- Cosine LR scheduler for smoother convergence
- Checkpoints every 500 steps — protects against session crashes
- Best model selected by Macro F1 on validation set

In [ ]:
# Class weights — QUESTION is 0.3% of data, needs strong boost
CLASS_WEIGHTS = torch.tensor([1.0, 4.0, 2.5, 40.0])

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fn = CrossEntropyLoss(
            weight=CLASS_WEIGHTS.to(logits.device),
            ignore_index=-100,
        )
        # NUM_LABELS constant avoids DataParallel .config AttributeError
        loss = loss_fn(
            logits.view(-1, NUM_LABELS),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    true_labels, pred_labels = [], []
    for pred_row, label_row in zip(predictions, labels):
        for p, l in zip(pred_row, label_row):
            if l != -100:
                true_labels.append(ID2LABEL[l])
                pred_labels.append(ID2LABEL[p])

    macro_f1 = f1_score(
        true_labels, pred_labels,
        labels=["COMMA", "PERIOD", "QUESTION"],
        average="macro",
    )
    # Per-class F1 printed every eval step
    for cls in ["COMMA", "PERIOD", "QUESTION"]:
        f1 = f1_score(true_labels, pred_labels, labels=[cls], average="macro")
        print(f"  {cls}: {f1:.4f}")
    return {"macro_f1": macro_f1}


training_args = TrainingArguments(
    output_dir="/kaggle/working/punct_model",
    lr_scheduler_type="cosine",         # cosine > linear for NER
    num_train_epochs=3,                 # fits within 9h Kaggle limit
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    evaluation_strategy="steps",
    eval_steps=500,                     # eval every ~30 min
    save_strategy="steps",
    save_steps=500,                     # checkpoint every ~30 min
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    report_to="none",
    save_total_limit=2,
    logging_steps=200,
    seed=SEED,
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

# Save best model right after training — prevents loss on session crash
MODEL_OUT = "/kaggle/working/punct_model_best"
trainer.save_model(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)
print(f"Model saved to {MODEL_OUT}")


Step,Training Loss,Validation Loss,Macro F1
500,0.510900,0.358869,0.670487
1000,0.340300,0.285433,0.710557
1500,0.301600,0.264991,0.737625
2000,0.261500,0.251068,0.748740
2500,0.254600,0.247718,0.754526
3000,0.239700,0.246579,0.748717
3500,0.213300,0.244361,0.756564
4000,0.219500,0.242502,0.758053


  COMMA: 0.6283
  PERIOD: 0.7488
  QUESTION: 0.6344
  COMMA: 0.6781
  PERIOD: 0.7907
  QUESTION: 0.6629
  COMMA: 0.6831
  PERIOD: 0.8023
  QUESTION: 0.7275
  COMMA: 0.7149
  PERIOD: 0.8099
  QUESTION: 0.7214
  COMMA: 0.6979
  PERIOD: 0.8154
  QUESTION: 0.7503
  COMMA: 0.7175
  PERIOD: 0.8230
  QUESTION: 0.7056
  COMMA: 0.7137
  PERIOD: 0.8263
  QUESTION: 0.7296
  COMMA: 0.7133
  PERIOD: 0.8234
  QUESTION: 0.7375
Model saved to /kaggle/working/punct_model_best


## Results

In [ ]:
df_test = pd.read_csv("/kaggle/input/datasets/daieeane/train6/test.csv")

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict_batch(texts, batch_size=64):
    all_predictions = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_words = [str(t).split() for t in batch_texts]

        encoding = tokenizer(
            batch_words,
            is_split_into_words=True,
            truncation=True,
            max_length=192,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoding)

        batch_preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()

        for idx, (words, preds) in enumerate(zip(batch_words, batch_preds)):
            word_ids   = encoding.word_ids(batch_index=idx)
            word_preds = {}
            for pos, word_id in enumerate(word_ids):
                if word_id is not None and word_id not in word_preds:
                    word_preds[word_id] = ID2LABEL[preds[pos]]

            preds_out = [word_preds.get(j, "O") for j in range(len(words))]
            all_predictions.append(" ".join(preds_out))

        if (i // batch_size) % 10 == 0:
            print(f"  Processed: {min(i + batch_size, len(texts))}/{len(texts)}")

    return all_predictions


print("Predicting...")
predictions = predict_batch(df_test["input_text"].tolist())

# Validate output lengths match input
mismatches = 0
for i, (text, pred) in enumerate(zip(df_test["input_text"], predictions)):
    expected = len(str(text).split())
    actual   = len(pred.split())
    if expected != actual:
        mismatches += 1
        print(f"Mismatch row {i}: expected {expected}, got {actual}")
print(f"Length mismatches: {mismatches}")

submission = pd.DataFrame({
    "id":     df_test["id"],
    "labels": predictions,
})
submission.to_csv("/kaggle/working/submission2.csv", index=False)
print(f"Done! Rows: {len(submission)}")
print(submission.head(3).to_string())


Predicting...
  Processed: 64/3552
  Processed: 704/3552
  Processed: 1344/3552
  Processed: 1984/3552
  Processed: 2624/3552
  Processed: 3264/3552
Length mismatches: 0
Done! Rows: 3552
          id                                                                 labels
0  kzp_00001                                           O O O O COMMA O O O O PERIOD
1  kzp_00002                           O O O O O O PERIOD O O O O PERIOD O O PERIOD
2  kzp_00003  O O COMMA O O O O O O O O O PERIOD O O O O O O O O O O O O O O PERIOD
